In [0]:
pip install langchain-google-genai python-dotenv tabulate delta-spark --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
# from pyspark.sql import SparkSession

In [0]:
df_btc_weekly = (spark.table("workspace.gold.tb_bitcoin_weekly")
           .orderBy("DT_WEEK_START", ascending=False)
           .limit(8)
           )
df_btc_weekly.display()

In [0]:
cols = df_btc_weekly.columns
rows = df_btc_weekly.collect()

header = "| " + " | ".join(cols) + " |"
separator = "| " + " | ".join(["---"] * len(cols)) + " |"
body = "\n".join("| " + " | ".join(str(v) for v in r) + " |" for r in rows)
df_weekly_markdown = "\n".join([header, separator, body])

In [0]:
print(df_weekly_markdown)

In [0]:
#read prompt from prompts folder
with open('PROMPTS/analysis_agent.txt') as file:
    prompt_file = file.read()

# substitui o placeholder pelos dados
prompt = prompt_file.replace('{dados_semanal}', df_weekly_markdown)
load_dotenv('./.env')
# chama o Gemini
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash-lite',
    google_api_key=os.getenv('GOOGLE_API_KEY')
)

resposta = llm.invoke(prompt)
analise = resposta.content

print(analise)

In [0]:
# load_dotenv('./.env')
# print(os.getenv('GOOGLE_API_KEY'))
# api_key = os.getenv("GOOGLE_API_KEY")

# print("GOOGLE_API_KEY existe:", bool(api_key))
# print("GOOGLE_API_KEY prefixo:", api_key[:5] if api_key else None)

# print("GOOGLE_GENAI_USE_VERTEXAI:",
#       os.getenv("GOOGLE_GENAI_USE_VERTEXAI"))

# print("GOOGLE_CLOUD_PROJECT:",
#       os.getenv("GOOGLE_CLOUD_PROJECT"))

# print("GOOGLE_APPLICATION_CREDENTIALS:",
#       os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))